In [1]:
!pip install -U transformers accelerate datasets evaluate rouge_score sacrebleu bert_score sentencepiece -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [37]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import T5Tokenizer, T5ForConditionalGeneration, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

from tqdm.auto import tqdm
import evaluate
from dotenv import load_dotenv
load_dotenv()

True

In [38]:
# df = pd.read_csv('/content/drive/MyDrive/t5_tune/two_problems/data/reviews_with_summaries_longer.csv')
df = pd.read_csv('data/reviews_with_summaries_longer.csv')
df.head()

,review,sentiment,review_clean,n_words,summary
0,I don't think I really have any spoilers in he...,positive,I don't think I really have any spoilers in he...,725,The review says this positive Western is easy ...
1,A magical journey concocted by Alexander Korda...,positive,A magical journey concocted by Alexander Korda...,160,The review is very positive because it praises...
2,"What an awesome movie! It was very scary, grea...",positive,"What an awesome movie! It was very scary, grea...",163,"This movie is awesome and very scary, with gre..."
3,I had a heck of a good time viewing this pictu...,positive,I had a heck of a good time viewing this pictu...,206,The reviewer liked the film because the acting...
4,Citizen X tells the real life drama of the sea...,positive,Citizen X tells the real life drama of the sea...,132,Citizen X is a great film that shows a long re...


## Смотрим как сейчас сгенерирует

In [39]:
model_name = "t5-base"

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"


t5_tokenizer = T5Tokenizer.from_pretrained(model_name)
t5_model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

print(device)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

mps


https://huggingface.co/docs/transformers/model_doc/t5#transformers.T5ForConditionalGeneration

In [ ]:
for i in range(10):
    example = df.iloc[i]

    text = "summarize movie review: " + example["review_clean"]
    true_summary = example["summary"]

    inputs = t5_tokenizer(text, return_tensors="pt").input_ids.to(device)

    with torch.no_grad():
        outputs = t5_model.generate(inputs, max_length=40, min_length=10, num_beams=4, early_stopping=True)

    t5_summary = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("Reference:")
    print(true_summary)

    print("\nt5 generated:")
    print(t5_summary)
    print('-------------------------------')
    print('\n\n')

Reference:
The review says this positive Western is easy to enjoy for everyone, with very likable characters, funny scenes, a good story, beautiful music and scenery, and exciting action.

t5 generated:
"Shadows in the Dust" is one of my favorite movies . it has a good story, very engaging characters, beautiful scenery and plenty of action balanced with humor and 
-------------------------------



Reference:
The review is very positive because it praises the magical story, beautiful Technicolor, a great Miklós Rózsa music score, strong acting, and clear vivid DVD colors.

t5 generated:
a magical journey concocted by Alexander Korda and Michael Powell . some of the most ravishing early Technicolor, a SUBLIME and a
-------------------------------



Reference:
This movie is awesome and very scary, with great acting, a well-written story and twists, interesting characters, good direction, and a surprise ending, so I highly recommend it.

t5 generated:
this is one popcorn horror flick tha

## Пробуем дообучить

In [41]:
df['n_words'].describe()

count    2000.000000
mean      246.281000
std       164.055073
min       100.000000
25%       137.750000
50%       186.000000
75%       296.000000
max      1808.000000
Name: n_words, dtype: float64

In [42]:
df['summary_length'] = df['summary'].apply(lambda x: len(x.split()))
df['summary_length'].describe()

count    2000.000000
mean       28.718500
std         2.684001
min        19.000000
25%        27.000000
50%        29.000000
75%        31.000000
max        38.000000
Name: summary_length, dtype: float64

In [43]:
df = df[df['summary_length'] > 3]
df['summary_length'].describe()

count    2000.000000
mean       28.718500
std         2.684001
min        19.000000
25%        27.000000
50%        29.000000
75%        31.000000
max        38.000000
Name: summary_length, dtype: float64

In [44]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["sentiment"])

train_df.shape, test_df.shape

((1600, 6), (400, 6))

## Учим

In [45]:
prefix = "summarize movie review: "

train_df["input_text"] = prefix + train_df["review_clean"]
test_df["input_text"] = prefix + test_df["review_clean"]

In [46]:
train_dataset = Dataset.from_pandas(train_df[["input_text", "summary"]])
test_dataset = Dataset.from_pandas(test_df[["input_text", "summary"]])

In [ ]:
def preprocess(examples):
    model_inputs = t5_tokenizer(examples["input_text"], max_length=600, padding="max_length", truncation=True)

    labels = t5_tokenizer(examples["summary"], max_length=60, padding="max_length", truncation=True)

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

https://github.com/hse-ds/iad-deep-learning/blob/master/2025/homeworks/homework_04.ipynb

https://habr.com/ru/articles/762140/

In [48]:
tokenized_dataset = train_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)
# tokenized_dataset.set_format("torch")
# test_dataset.set_format("torch")

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [49]:
data_collator = DataCollatorForSeq2Seq(tokenizer=t5_tokenizer, model=t5_model)

In [50]:
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("sacrebleu")
bertscore_metric = evaluate.load("bertscore")

In [51]:
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    preds = [pred if pred != "" else "empty" for pred in preds]

    return preds, labels

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = t5_tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, t5_tokenizer.pad_token_id)

    decoded_labels = t5_tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    bleu_result = bleu_metric.compute(predictions=decoded_preds, references=[[label] for label in decoded_labels])
    rouge_result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    bertscore_result = bertscore_metric.compute(predictions=decoded_preds, references=decoded_labels, lang="en")

    prediction_lens = [np.count_nonzero(pred != t5_tokenizer.pad_token_id) for pred in preds]

    result = {
        "bleu": bleu_result["score"],
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bertscore_precision": np.mean(bertscore_result["precision"]),
        "bertscore_recall": np.mean(bertscore_result["recall"]),
        "bertscore_f1": np.mean(bertscore_result["f1"]),
        "gen_len": np.mean(prediction_lens),
    }

    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
new_model = "t5_base"

training_args = Seq2SeqTrainingArguments(
    output_dir=new_model,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=10,
    predict_with_generate=True,
    generation_max_length=60,
    fp16=True,
    push_to_hub=False
)

trainer = Seq2SeqTrainer(
    model=t5_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_dataset)

test_metrics

## Инферим немного

In [54]:
model_path = "models/t5_base_summarizer"


t5_tokenizer = T5Tokenizer.from_pretrained(model_path)
t5_model = T5ForConditionalGeneration.from_pretrained(model_path).to(device)


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

### Base tuned

In [ ]:
for i in range(10):
    example = df.iloc[1800 + i]

    text = "summarize movie review: " + example["review_clean"]
    true_summary = example["summary"]

    inputs = t5_tokenizer(text, return_tensors="pt").input_ids.to(device)

    with torch.no_grad():
        outputs = t5_model.generate(inputs, max_length=60, min_length=10, num_beams=4, early_stopping=True)

    t5_summary = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("Reference:")
    print(true_summary)

    print("\nt5 generated:")
    print(t5_summary)
    print('-------------------------------')
    print('\n\n')

Reference:
The review is very negative, saying the show is a bad sketch and tries too hard to be funny, with weak writing and an unfunny host, wasting viewers’ time.

t5 generated:
The reviewer says the show is a crap show, with hard sketches, no funny sketches, and a no-talent host, so they give it a fair shot.
-------------------------------



Reference:
The review says this sixth horror movie is boring and has a slow pace, dumb ending, bland acting, and unpleasant revelations, while the mirror causes the strange but not exciting events.

t5 generated:
The review is negative because the horror is dull and lifeless, with ropey acting and Ross Partridge’s huge hair not helping matters.
-------------------------------



Reference:
The reviewer hates the film because it has unrealistic drug-life stereotypes, a confusing shoe quest with no clear plot, and characters feel too outrageous for real life.

t5 generated:
The reviewer hated the film because it has shallow Hollywood fare and st

### Small tuned

In [56]:
model_path = "models/t5_small_summarizer"

t5_tokenizer = T5Tokenizer.from_pretrained(model_path)
t5_model = T5ForConditionalGeneration.from_pretrained(model_path).to(device)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
for i in range(10):
    example = df.iloc[1800 + i]

    text = "summarize movie review: " + example["review_clean"]
    true_summary = example["summary"]

    inputs = t5_tokenizer(text, return_tensors="pt").input_ids.to(device)

    with torch.no_grad():
        outputs = t5_model.generate(inputs, max_length=60, min_length=10, num_beams=4, early_stopping=True)

    t5_summary = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("Reference:")
    print(true_summary)

    print("\nt5 generated:")
    print(t5_summary)
    print('-------------------------------')
    print('\n\n')

Reference:
The review is very negative, saying the show is a bad sketch and tries too hard to be funny, with weak writing and an unfunny host, wasting viewers’ time.

t5 generated:
The reviewer says the show is a crap show because Spike is not funny, his sketches are hard, and the writers are struggling to write material for a no-talent host.
-------------------------------





[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (816 > 512). Running this sequence through the model will result in indexing errors


Reference:
The review says this sixth horror movie is boring and has a slow pace, dumb ending, bland acting, and unpleasant revelations, while the mirror causes the strange but not exciting events.

t5 generated:
The review is negative because the horror is dull and lifeless, the acting and hairdo Ross Partridge’s hair are not helping matters.
-------------------------------



Reference:
The reviewer hates the film because it has unrealistic drug-life stereotypes, a confusing shoe quest with no clear plot, and characters feel too outrageous for real life.

t5 generated:
The reviewer says the film is very bad because of its shallow Hollywood fare and stereotypical caricatures, realism, and a lack of shoes, so they decided to end it.
-------------------------------



Reference:
The review is very negative: the film was boring, the CGI ship and creature looked bad, and the writer says to avoid it and not buy the DVD, with a shame about who funded it.

t5 generated:
The review says the m

In [63]:
def eval_(model_path, test_df):
    tokenizer = T5Tokenizer.from_pretrained(model_path)
    model = T5ForConditionalGeneration.from_pretrained(model_path).to(device)
    model.eval()

    t5_summaries = []
    reference_summaries = []
    prediction_lens = []

    for i in tqdm(range(len(test_df))):

        true_summary = test_df.iloc[i]["summary"]
        input_text = test_df.iloc[i]["input_text"]

        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).input_ids.to(device)

        with torch.no_grad():
            outputs = model.generate(inputs, max_length=60, min_length=10, num_beams=4, early_stopping=True)

        t5_summary = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

        t5_summaries.append(t5_summary)
        reference_summaries.append(true_summary)
        prediction_lens.append(np.count_nonzero(outputs[0].cpu().numpy() != tokenizer.pad_token_id))

    bleu_result = bleu_metric.compute(predictions=t5_summaries, references=[[label] for label in reference_summaries])
    rouge_result = rouge_metric.compute(predictions=t5_summaries, references=reference_summaries)
    bertscore_result = bertscore_metric.compute(predictions=t5_summaries, references=reference_summaries, lang="en")

    result = {
        "bleu": bleu_result["score"],
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bertscore_precision": np.mean(bertscore_result["precision"]),
        "bertscore_recall": np.mean(bertscore_result["recall"]),
        "bertscore_f1": np.mean(bertscore_result["f1"]),
        "gen_len": np.mean(prediction_lens),
    }

    return {k: round(float(v), 4) for k, v in result.items()}

In [64]:
small_metrics = eval_("models/t5_small_summarizer", test_df)
base_metrics = eval_("models/t5_base_summarizer", test_df)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]

In [65]:
comparison_df = pd.DataFrame([{"model": "T5-small fine-tuned", **small_metrics}, {"model": "T5-base fine-tuned", **base_metrics}])
comparison_df

,model,bleu,rouge1,rouge2,rougeL,bertscore_precision,bertscore_recall,bertscore_f1,gen_len
0,T5-small fine-tuned,14.1498,0.4412,0.1823,0.3544,0.9050,0.9076,0.9062,42.775
1,T5-base fine-tuned,15.5086,0.4607,0.1934,0.3696,0.9114,0.9125,0.9119,42.020


## Смотрим базовые модели

In [61]:
small_zero_metrics = eval_("t5-small", test_df)
base_zero_metrics = eval_("t5-base", test_df)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

  0%|          | 0/400 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]

In [62]:
comparison_df = pd.DataFrame([{"model": "T5-small", **small_zero_metrics}, {"model": "T5-base", **base_zero_metrics}])
comparison_df

,model,bleu,rouge1,rouge2,rougeL,bertscore_precision,bertscore_recall,bertscore_f1,gen_len
0,T5-small,3.5236,0.2639,0.0654,0.1904,0.8574,0.8723,0.8647,53.1500
1,T5-base,3.4563,0.2798,0.0741,0.1988,0.8522,0.8748,0.8633,52.4925
